# Combined Pipeline — ASFormer (2-class) + CatBoost (5-class phases)

**Stage 1 — Segmentor.** Best ASFormer config from `results/hypertune/asformer_MPW`
(2-class: `phase=0`, `nonphase=1`; encoder = Identity, MPW full 33 joints). Per fold, the
notebook **loads `best_encoder.pt` / `best_segmentor.pt` if they exist, otherwise trains**
from the tuned config.

**Stage 2 — Classifier.** Tuned CatBoost (your best params) on **MPW Bottom Half (10J,
indices 23-32)**. Trained per fold on that fold's *train* phase-frames only, with balanced
class weights and **no `eval_set`** so the val set stays clean.

**Compose (5-class).** For each val video: run the segmentor → extract predicted phase
segments (`pred==0`) → classify each segment's bottom-half frames with CatBoost →
majority-vote a phase (0-3) for the segment. Frames the segmentor calls `nonphase` become
class 4. Result is a per-frame label in `{Phase1..Phase4, nonphase}`.

**Metrics** (val set, averaged over folds 1-4; `fold_0` excluded because it tuned the HPs):
F1@IoU{0.1,0.25,0.5} (macro over the 4 phase classes), normalized edit distance, frame
accuracy, and a 5-class macro-F1 confusion matrix.

> Heavy if checkpoints are absent: it trains ASFormer per fold (early-stops on val_f1).
> Point `SEG_CKPT_DIR` at an existing trained run to skip training.

In [ ]:
import os, sys, json, importlib.util
import numpy as np
import h5py
import torch
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Locate project root (server path first, else two levels up from cwd) ──────
CANDIDATES = ["/code/jjiang23/BalanceTestThesis",
              os.path.abspath(os.path.join(os.getcwd(), ".."))]
PROJECT_ROOT = next((c for c in CANDIDATES if os.path.isdir(os.path.join(c, "src"))),
                    CANDIDATES[0])
for p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("PROJECT_ROOT:", PROJECT_ROOT)

from data.PoseDataset import PoseDataset, load_video_h5
from Trainer import Trainer
from utils.eval.metric_utils import (
    predict_video, extract_segments, compute_averaged_video_metrics,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SPLITS_PATH = "/code/jjiang23/pathml/aim2_balanceV2/data/splits.json"
if not os.path.exists(SPLITS_PATH):
    SPLITS_PATH = os.path.join(PROJECT_ROOT, "splits.json")

H5_KEY       = "world_mp_cropped_iou"
BOTTOM_HALF  = list(range(23, 33))          # MPW bottom half (10 joints) for CatBoost
IOU          = (0.1, 0.25, 0.5)
FPS          = 30
EXCLUDE_FOLD = "fold_0"                      # tuned HPs -> contaminated, drop from eval
PHASE_NAMES5 = ["Phase1", "Phase2", "Phase3", "Phase4", "nonphase"]

# Data config for the segmentor (full 33 joints; joint_indices absent -> None)
D_CFG = dict(window_size=120, stride=60, h5_key=H5_KEY, num_joints=33)

# Best ASFormer segmentor (results/hypertune/asformer_MPW)
S_CFG = dict(
    num_classes          = 2,
    num_f_maps           = 64,
    num_layers           = 7,
    num_decoders         = 3,
    r1                   = 4,
    r2                   = 4,
    channel_masking_rate = 0.20632429079356646,
    att_type             = "block_att",
    init_segmentor_path  = os.path.join(PROJECT_ROOT, "initializers/segementor/ASFormer.py"),
)
T_CFG = dict(
    epochs               = 1000,
    batch_size           = 32,
    lr                   = 0.0002038185867361128,
    weight_decay         = 5.257723195774366e-05,
    early_stop_patience  = 20,
    early_stop_min_delta = 0.001,
    lambda_smooth        = 0.01557,
    time_alignment       = "upsample_preds",
    early_stop_monitor   = "val_f1",
)

# Per-fold segmentor checkpoints live here. If best_*.pt exist they are loaded;
# otherwise the fold is trained and written here. Point this at an existing run
# (must contain <fold>/best_encoder.pt & best_segmentor.pt) to skip training.
SEG_CKPT_DIR = os.path.join(PROJECT_ROOT,
                            "results/Identity/MPW/asf_hyper/asf_hyper/20260622_191332")

# Your best CatBoost params (tuned on MPW bottom half / fold_0)
CATBOOST_PARAMS = dict(
    loss_function   = "MultiClass",
    iterations      = 1000,
    depth           = 10,
    learning_rate   = 0.009138119402547074,
    l2_leaf_reg     = 2.6294307360356752,
    random_strength = 2.549160677013768,
    border_count    = 64,
    random_seed     = 42,
    thread_count    = -1,
    verbose         = 0,
)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

EVAL_FOLDS = [fn for fn in splits.keys() if fn != EXCLUDE_FOLD]
print("Splits:", list(splits.keys()))
print("Eval folds:", EVAL_FOLDS, " (excluded:", EXCLUDE_FOLD, ")")
print("Segmentor ckpt dir:", SEG_CKPT_DIR)

In [ ]:
# ── Builders for encoder + segmentor (reuse repo initializers) ───────────────

def _load_module(path):
    spec = importlib.util.spec_from_file_location("_dyn", path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

_ENC_INIT = _load_module(os.path.join(PROJECT_ROOT, "initializers/encoder/Identity.py")
                         ).initialize_encoder
_SEG_INIT = _load_module(S_CFG["init_segmentor_path"]).initialize_segmentor


def build_models(class_weights=None):
    encoder = _ENC_INIT(D_CFG, None)
    segmentor = _SEG_INIT(
        S_CFG, encoder,
        class_weights=class_weights,
        lambda_smooth=T_CFG["lambda_smooth"],
        time_alignment=T_CFG["time_alignment"],
    )
    return encoder, segmentor


def get_fold_segmentor(fold_name, train_files, val_files):
    """Load per-fold checkpoints if present, else train from the tuned config."""
    fold_dir = os.path.join(SEG_CKPT_DIR, fold_name)
    enc_ckpt = os.path.join(fold_dir, "best_encoder.pt")
    seg_ckpt = os.path.join(fold_dir, "best_segmentor.pt")

    if os.path.exists(enc_ckpt) and os.path.exists(seg_ckpt):
        encoder, segmentor = build_models(class_weights=None)
        encoder.load_state_dict(torch.load(enc_ckpt, map_location=DEVICE))
        segmentor.load_state_dict(torch.load(seg_ckpt, map_location=DEVICE), strict=False)
        print(f"  [{fold_name}] loaded checkpoints")
    else:
        print(f"  [{fold_name}] no checkpoints -> training ASFormer (this is slow)")
        train_ds = PoseDataset(train_files, featureH5Key=H5_KEY,
                               window_size=D_CFG["window_size"], stride=D_CFG["stride"],
                               augment=True, joint_indices=None)
        val_ds   = PoseDataset(val_files, featureH5Key=H5_KEY,
                               window_size=D_CFG["window_size"], stride=D_CFG["stride"],
                               augment=False, joint_indices=None)
        class_weights = train_ds.compute_class_weights(device=DEVICE)
        encoder, segmentor = build_models(class_weights=class_weights)
        trainer = Trainer(encoder, segmentor,
                          early_stop_patience=T_CFG["early_stop_patience"],
                          early_stop_min_delta=T_CFG["early_stop_min_delta"],
                          early_stop_monitor=T_CFG["early_stop_monitor"])
        os.makedirs(fold_dir, exist_ok=True)
        trainer.train(save_dir=fold_dir, batch_gen=train_ds, val_batch_gen=val_ds,
                      num_epochs=T_CFG["epochs"], batch_size=T_CFG["batch_size"],
                      learning_rate=float(T_CFG["lr"]),
                      weight_decay=float(T_CFG["weight_decay"]), device=str(DEVICE))
        encoder.load_state_dict(torch.load(enc_ckpt, map_location=DEVICE))
        segmentor.load_state_dict(torch.load(seg_ckpt, map_location=DEVICE), strict=False)

    encoder.to(DEVICE).eval()
    segmentor.to(DEVICE).eval()
    return encoder, segmentor

In [ ]:
# ── CatBoost stage: bottom-half phase-frame loader + per-fold trainer ────────

def load_phase_frames_bh(files):
    """Bottom-half (10J) phase-only frames -> X (N, 30), y (N,) in {0..3}."""
    Xs, ys = [], []
    for path in files:
        try:
            kps, labels = load_video_h5(path, H5_KEY, allPhases=True, joint_indices=BOTTOM_HALF)
        except Exception as e:
            print(f"    skip {os.path.basename(path)}: {e}")
            continue
        T, J, D = kps.shape
        X = kps.reshape(T, J * D)
        m = labels < 4
        if m.sum() == 0:
            continue
        Xs.append(X[m]); ys.append(labels[m].astype(np.int32))
    if not Xs:
        return np.empty((0, len(BOTTOM_HALF) * 3), np.float32), np.empty((0,), np.int32)
    return np.concatenate(Xs), np.concatenate(ys)


def train_fold_catboost(train_files):
    X, y = load_phase_frames_bh(train_files)
    sw = compute_sample_weight('balanced', y)
    clf = CatBoostClassifier(**CATBOOST_PARAMS)
    clf.fit(X, y, sample_weight=sw)          # no eval_set -> val set untouched
    return clf

In [ ]:
# ── Combined per-video inference: segmentor -> segments -> CatBoost -> 5-class ─

def predict_video_5class(h5_path, encoder, segmentor, clf):
    # 5-class GT + bottom-half features (frame-aligned)
    kps_bh, gt5 = load_video_h5(h5_path, H5_KEY, allPhases=True, joint_indices=BOTTOM_HALF)
    feats_bh = kps_bh.reshape(kps_bh.shape[0], -1)          # (T, 30)

    # 2-class segmentor prediction over full-joint video (phase=0, nonphase=1)
    _, seg_pred, _ = predict_video(h5_path, encoder, segmentor, D_CFG, DEVICE,
                                   stride_override=D_CFG["stride"])

    T = min(len(seg_pred), len(gt5))
    seg_pred, gt5, feats_bh = seg_pred[:T], gt5[:T], feats_bh[:T]

    pred5 = np.full(T, 4, dtype=np.int64)                   # default: nonphase
    for (s, e) in extract_segments(seg_pred, class_id=0):   # predicted phase runs
        fr = feats_bh[s:e + 1]
        if len(fr) == 0:
            continue
        cls = clf.predict(fr).reshape(-1).astype(np.int64)  # per-frame phase 0..3
        pred5[s:e + 1] = np.bincount(cls, minlength=4).argmax()   # majority vote
    return gt5, pred5

In [ ]:
# ── Run all eval folds ───────────────────────────────────────────────────────

def fold_metric_row(gt_list, pred_list):
    # class-agnostic: frame accuracy + normalized edit (collapse of full 5-class seq)
    base = compute_averaged_video_metrics(pred_list, gt_list, class_id=0,
                                          iou_thresholds=IOU, fps=FPS, ignore_index=-100)
    acc   = base["frame_accuracy"]
    nedit = base["edit_distance_normalized"]

    # F1@IoU macro-averaged over the 4 phase classes
    f1_by_thr = {thr: [] for thr in IOU}
    for c in (0, 1, 2, 3):
        m = compute_averaged_video_metrics(pred_list, gt_list, class_id=c,
                                           iou_thresholds=IOU, fps=FPS, ignore_index=-100)
        for thr in IOU:
            v = m[f"f1_iou_{thr}"]
            if v is not None:
                f1_by_thr[thr].append(v)
    f1_macro = {thr: (float(np.mean(f1_by_thr[thr])) if f1_by_thr[thr] else np.nan)
                for thr in IOU}

    # frame-level 5-class macro F1
    yt = np.concatenate(gt_list); yp = np.concatenate(pred_list)
    macro_f1 = f1_score(yt, yp, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)
    return acc, nedit, f1_macro, macro_f1, yt, yp


fold_rows   = []
pooled_yt   = []
pooled_yp   = []

for fold_name in EVAL_FOLDS:
    print(f"\n{'='*66}\nFOLD {fold_name}\n{'='*66}")
    fd = splits[fold_name]

    encoder, segmentor = get_fold_segmentor(fold_name, fd["train"], fd["val"])
    clf = train_fold_catboost(fd["train"])

    gts, preds = [], []
    for v in fd["val"]:
        try:
            g, p = predict_video_5class(v, encoder, segmentor, clf)
        except Exception as e:
            print(f"    skip {os.path.basename(v)}: {e}")
            continue
        gts.append(g); preds.append(p)

    acc, nedit, f1m, macro_f1, yt, yp = fold_metric_row(gts, preds)
    pooled_yt.append(yt); pooled_yp.append(yp)
    fold_rows.append({
        "Fold": fold_name, "N_val": len(gts),
        "F1@0.10": f1m[0.1], "F1@0.25": f1m[0.25], "F1@0.50": f1m[0.5],
        "NormEdit": nedit, "Acc": acc, "MacroF1_5cls": macro_f1,
    })
    print(f"  F1@IoU: {f1m[0.1]:.3f}/{f1m[0.25]:.3f}/{f1m[0.5]:.3f}  "
          f"NormEdit={nedit:.2f}  Acc={acc:.3f}  MacroF1(5)={macro_f1:.3f}")

    del encoder, segmentor, clf
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# ── Summary table (per fold + mean +/- std) ──────────────────────────────────
df = pd.DataFrame(fold_rows).set_index("Fold")
metric_cols = ["F1@0.10", "F1@0.25", "F1@0.50", "NormEdit", "Acc", "MacroF1_5cls"]

summary = df[metric_cols].agg(['mean', 'std'])
print("Per-fold metrics (validation set):\n")
print(df.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nAcross-fold mean +/- std:\n")
for c in metric_cols:
    print(f"  {c:<14} {summary.loc['mean', c]:.4f} +/- {summary.loc['std', c]:.4f}")

df_out = df.copy()
df_out.loc["MEAN"] = df[metric_cols].mean()
df_out.loc["STD"]  = df[metric_cols].std()
df_out

In [ ]:
# ── 5-class confusion matrix (pooled over eval folds) + macro-F1 ─────────────
yt_all = np.concatenate(pooled_yt)
yp_all = np.concatenate(pooled_yp)

cm     = confusion_matrix(yt_all, yp_all, labels=[0, 1, 2, 3, 4])
cm_n   = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
macro  = f1_score(yt_all, yp_all, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=True,
            xticklabels=PHASE_NAMES5, yticklabels=PHASE_NAMES5, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'ASFormer + CatBoost — 5-class (pooled folds {EVAL_FOLDS})\n'
             f'Macro-F1 = {macro:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart: mean metrics across folds ─────────────────────────────────────
means = df[metric_cols].mean()
stds  = df[metric_cols].std()
# NormEdit is 0-100; scale to 0-1 for a shared axis, label it explicitly
plot_means = means.copy(); plot_stds = stds.copy()
plot_means["NormEdit"] = means["NormEdit"] / 100.0
plot_stds["NormEdit"]  = stds["NormEdit"] / 100.0

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(metric_cols))
bars = ax.bar(x, plot_means.values, yerr=plot_stds.values, capsize=4,
              color='#4C72B0', alpha=0.85)
for b, c in zip(bars, metric_cols):
    raw = means[c]
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{raw:.3f}' if c != 'NormEdit' else f'{raw:.1f}',
            ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(["F1@0.1", "F1@0.25", "F1@0.5", "NormEdit/100", "Acc", "MacroF1(5)"],
                   rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Combined ASFormer + CatBoost — mean metrics (val, folds 1-4)')
plt.tight_layout()
plt.show()